[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [GeoPandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)

# Measuring Distance and Area &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import pyproj
import matplotlib.pyplot as plt

COUNTRIES_URL = ("https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/data/nordic-countries.geojson")
local_copy = Path("../../data/nordic-countries.geojson")
countries = gpd.read_file(local_copy if local_copy.exists() else COUNTRIES_URL)


weather_stations = pd.DataFrame({
    "name": ["Bergen", "Oslo", "Svalbard", "Tromso", "Kirkenes"],
    "latitude": [60.39, 59.91, 78.22, 69.65, 69.73],
    "longitude": [5.32, 10.75, 15.65, 18.96, 30.05],
})


def mainland_norway():
    """Norway's outline south of 74 degrees north, without Svalbard, in UTM zone 33N."""
    parts = countries[countries["ADMIN"] == "Norway"].to_crs(25833).explode()
    return parts[parts.geometry.bounds["maxy"] < 8_200_000].union_all()


def station_network(n=300, seed=2026):
    """A made-up register of weather stations on mainland Norway, one row each."""
    rng = np.random.default_rng(seed)
    land = mainland_norway()
    x_min, y_min, x_max, y_max = land.bounds
    east, north = [], []
    while len(east) < n:
        x, y = rng.uniform(x_min, x_max), rng.uniform(y_min, y_max)
        if land.contains(shapely.Point(x, y)):
            east.append(x)
            north.append(y)
    lonlat = gpd.GeoSeries(gpd.points_from_xy(east, north), crs=25833).to_crs(4326)
    return pd.DataFrame({
        "station": [f"NS{i:03d}" for i in range(1, n + 1)],
        "latitude": lonlat.y.round(4).to_numpy(), "longitude": lonlat.x.round(4).to_numpy(),
        "elevation_m": np.clip(rng.gamma(1.6, 220, n), 2, 1900).round(0).astype(int),
        "owner": rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2]),
        "installed": rng.integers(1950, 2025, n),
    })


DISTRICT_CENTERS = {"South": (8.0, 58.6), "East": (10.8, 60.5), "West": (6.0, 60.8), "Mid": (11.0, 63.6),
                    "North": (14.5, 67.0), "Far North": (19.0, 69.4), "Northeast": (25.5, 70.0),
                    "Svalbard": (16.0, 78.5)}


def forecast_districts():
    """Eight made-up forecast districts that cover Norway, in UTM zone 33N."""
    norway = countries[countries["ADMIN"] == "Norway"].to_crs(25833).geometry.iloc[0]
    centers = gpd.GeoSeries(gpd.points_from_xy(*zip(*DISTRICT_CENTERS.values())), crs=4326).to_crs(25833)
    cells = shapely.voronoi_polygons(shapely.MultiPoint(list(centers)),
                                     extend_to=shapely.box(*norway.buffer(200_000).bounds)).geoms
    names = [next(name for name, center in zip(DISTRICT_CENTERS, centers) if cell.contains(center))
             for cell in cells]
    districts = gpd.GeoDataFrame({"district": names}, geometry=[cell.intersection(norway) for cell in cells],
                                 crs=25833)
    return districts.set_index("district").loc[list(DISTRICT_CENTERS)].reset_index()


network = station_network()
stations = gpd.GeoDataFrame(
    network, geometry=gpd.points_from_xy(network["longitude"], network["latitude"]), crs=4326
)


named = gpd.GeoDataFrame(
    weather_stations,
    geometry=gpd.points_from_xy(weather_stations["longitude"], weather_stations["latitude"]),
    crs=4326,
)


districts = forecast_districts()
geod = pyproj.Geod(ellps="WGS84")
print("geopandas", gpd.__version__, "| pyproj", pyproj.__version__, "|", len(stations), "made-up stations,",
      len(districts), "made-up districts")

geopandas 1.2.0 | pyproj 3.8.0 | 300 made-up stations, 8 made-up districts


Each answer runs on its own after Setup. The stations and districts are made up, and the outlines are Natural Earth's.

**1.** Print Norway's boundary length in kilometers in UTM zone 33N and on the ellipsoid, with `geod.geometry_area_perimeter`.

In [2]:
norway = countries[countries["ADMIN"] == "Norway"]
planar = norway.to_crs(25833).length.iloc[0] / 1000
area, perimeter = geod.geometry_area_perimeter(shapely.orient_polygons(norway.geometry.iloc[0]))
print("UTM zone 33N:", round(planar), "km | ellipsoid:", round(perimeter / 1000), "km")

UTM zone 33N: 14937 km | ellipsoid: 14927 km


The boundary includes every island's coastline in the 1:50m outline, and the two lengths differ by about 10 km in almost 15,000.

**2.** Print the geodesic distance from Tromso to each of the other named stations, in kilometers to one decimal place.

In [3]:
tromso = named.iloc[3]
_, _, meters = geod.inv(np.full(5, tromso["longitude"]), np.full(5, tromso["latitude"]),
                        named["longitude"].to_numpy(), named["latitude"].to_numpy())
pd.Series((meters / 1000).round(1), index=named["name"]).drop("Tromso")

name
Bergen      1210.1
Oslo        1151.6
Svalbard     961.5
Kirkenes     429.3
dtype: float64

`geod.inv` takes whole arrays, so one call measures from Tromso to all five, and `drop` removes Tromso's distance to itself.

**3.** Measure every district's area in EPSG:3035 and print it beside the area in UTM zone 33N.

In [4]:
pd.DataFrame({
    "district": districts["district"],
    "utm_km2": (districts.area / 1e6).round(0).astype(int),
    "laea_km2": (districts.to_crs(3035).area / 1e6).round(0).astype(int),
})

,district,utm_km2,laea_km2
0,South,36056,35919
1,East,62504,62454
2,West,53904,53681
3,Mid,61575,61559
4,North,30984,31007
5,Far North,31283,31276
6,Northeast,48523,48360
7,Svalbard,62068,62099


EPSG:3035 is an equal-area projection, and its figures differ from UTM's by less than half a percent, as the ellipsoid's did in the worked examples.

**4.** Find the network station whose distance to Kirkenes differs most between UTM zone 33N and the ellipsoid, and print the gap in kilometers.

In [5]:
kirkenes = named.iloc[4]
planar = stations.to_crs(25833).distance(named.to_crs(25833).geometry.iloc[4]) / 1000
_, _, meters = geod.inv(np.full(len(stations), kirkenes["longitude"]),
                        np.full(len(stations), kirkenes["latitude"]),
                        stations["longitude"].to_numpy(), stations["latitude"].to_numpy())
gap = (planar - meters / 1000).abs()
worst = gap.idxmax()
print(stations.loc[worst, "station"], round(gap.max(), 1), "km |", round(stations.loc[worst, "longitude"], 2),
      "degrees east | scale", round(pyproj.Proj(25833).get_factors(stations.loc[worst, "longitude"],
                                                              stations.loc[worst, "latitude"]).meridional_scale, 4))

NS021 1.5 km | 5.32 degrees east | scale 1.0033


The largest gap, 1.5 km, is larger than the 1.0 km measured from Oslo. NS021 lies near Bergen, at 5.32 degrees east, where UTM zone 33N stretches lengths by 0.33%, and Kirkenes, at 30.05 degrees east, by 0.37%, so both ends of the line are far from the zone's central meridian.

**5.** Count the network stations that lie within 100 km of their own district's representative point. Join the stations to the districts, then look up each row's representative point by district name.

In [6]:
joined = gpd.sjoin(stations.to_crs(districts.crs), districts, how="left")
inside = districts.set_index("district").representative_point()
targets = gpd.GeoSeries(inside.loc[joined["district"]].to_numpy(), index=joined.index,
                        crs=districts.crs)
print((joined.distance(targets) <= 100_000).sum(), "of", len(joined))

160 of 300


The join gives each station its district, and the points are lined up row by row with the stations, so `distance` compares each station with its own district's point.

---

&#8592; **Back to:** [Measuring Distance and Area](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/06-measuring-distance-and-area.ipynb)  &nbsp;&middot;&nbsp;  [GeoPandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)
